# Projected Hessian Learning (PHL)

## Notebook Usage

This notebook trains ANI-style models with Hessian-vector-product (HVP) supervision for fast curvature-aware learning.

### Before Running
- The recommended python version is 3.12
- Install required packages: torchani==2.2.4, torch, wandb, numpy, tqdm, and h5py.
- Replace TorchANI source files with the repository versions of utils.py and data/\_\_init\_\_.py.
- The dataset used in this work can be downloaded from OpenREACT-CHON-EFH - Open REaction Dataset of Atomic ConfiguraTions comprising C, H, O, N with Energies, Forces, and Hessians on Figshare: https://doi.org/10.6084/m9.figshare.29189858.
- Set dspath to the downloaded dataset location.

### Recommended Execution Order
1. Run all setup and model-definition cells from top to bottom.
2. Verify split and training parameters before launching training.
3. Run the training cell in the Training section.


In [ ]:
import torch
import h5py
import torchani
import os
import math
from tqdm import tqdm
import random
import numpy as np
from IPython.display import clear_output
import wandb
import time

### Loading the model

In [ ]:
# Set seeds for reproducibility (seed: 7289038)
random.seed(7289038)
np.random.seed(7289038)
torch.manual_seed(7289038)

# Set the default dtype for tensors
torch.set_default_dtype(torch.float64)

# helper function to convert energy unit from Hartree to kcal/mol
from torchani.units import hartree2kcalmol

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Rcr = 5.2000e+00
Rca = 3.5000e+00
EtaR = torch.tensor([1.6000000e+01], device=device)
ShfR = torch.tensor([9.0000000e-01, 1.1687500e+00, 1.4375000e+00, 1.7062500e+00, 1.9750000e+00, 2.2437500e+00, 2.5125000e+00, 2.7812500e+00, 3.0500000e+00, 3.3187500e+00, 3.5875000e+00, 3.8562500e+00, 4.1250000e+00, 4.3937500e+00, 4.6625000e+00, 4.9312500e+00], device=device)
Zeta = torch.tensor([3.2000000e+01], device=device)
ShfZ = torch.tensor([1.9634954e-01, 5.8904862e-01, 9.8174770e-01, 1.3744468e+00, 1.7671459e+00, 2.1598449e+00, 2.5525440e+00, 2.9452431e+00], device=device)
EtaA = torch.tensor([8.0000000e+00], device=device)
ShfA = torch.tensor([9.0000000e-01, 1.5500000e+00, 2.2000000e+00, 2.8500000e+00], device=device)
species_order = ['H', 'C', 'N', 'O']
num_species = len(species_order)
aev_computer = torchani.AEVComputer(Rcr, Rca, EtaR, ShfR, EtaA, Zeta, ShfA, ShfZ, num_species)
energy_shifter = torchani.utils.EnergyShifter(None)


try:
    path = os.path.dirname(os.path.realpath(__file__))
except NameError:
    path = os.getcwd()
dspath = os.path.join(path, 'path/to/dataset.h5')

config = {"max_epochs":5000, "batch_size":400}

In [ ]:
# Change the proportions of the training, validation, and test sets as needed
training, skip, validation, test = torchani.data.load(dspath, additional_properties=('forces','hessian')
    ).subtract_self_energies(energy_shifter, species_order).species_to_indices(species_order).shuffle().split(0.8, 0.0, 0.1, None)

training = training.collate(config["batch_size"]).cache()
validation = validation.collate(config["batch_size"]).cache()
test = test.collate(config["batch_size"]).cache()

print('Self atomic energies: ', energy_shifter.self_energies)

In [ ]:
aev_dim = aev_computer.aev_length

H_network = torch.nn.Sequential(
    torch.nn.Linear(aev_dim, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 64),
    torch.nn.CELU(0.1),
    torch.nn.Linear(64, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 1),
)

C_network = torch.nn.Sequential(
    torch.nn.Linear(aev_dim, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 64),
    torch.nn.CELU(0.1),
    torch.nn.Linear(64, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 1),
)

N_network = torch.nn.Sequential(
    torch.nn.Linear(aev_dim, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 64),
    torch.nn.CELU(0.1),
    torch.nn.Linear(64, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 1),
)

O_network = torch.nn.Sequential(
    torch.nn.Linear(aev_dim, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 64),
    torch.nn.CELU(0.1),
    torch.nn.Linear(64, 256),
    torch.nn.CELU(0.1),
    torch.nn.Linear(256, 1),
)

nn = torchani.ANIModel([H_network, C_network, N_network, O_network])
print(nn)

In [ ]:
def init_params(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight, a=1.0)
        torch.nn.init.zeros_(m.bias)


nn.apply(init_params)

In [ ]:
model = torchani.nn.Sequential(aev_computer, nn).to(device)

In [ ]:
AdamW = torch.optim.AdamW([
    # H networks
    {'params': [H_network[0].weight]},
    {'params': [H_network[2].weight]},
    {'params': [H_network[4].weight]},
    {'params': [H_network[6].weight]},
    # C networks
    {'params': [C_network[0].weight]},
    {'params': [C_network[2].weight]},
    {'params': [C_network[4].weight]},
    {'params': [C_network[6].weight]},
    # N networks
    {'params': [N_network[0].weight]},
    {'params': [N_network[2].weight]},
    {'params': [N_network[4].weight]},
    {'params': [N_network[6].weight]},
    # O networks
    {'params': [O_network[0].weight]},
    {'params': [O_network[2].weight]},
    {'params': [O_network[4].weight]},
    {'params': [O_network[6].weight]}
])

SGD = torch.optim.SGD([
    # H networks
    {'params': [H_network[0].bias]},
    {'params': [H_network[2].bias]},
    {'params': [H_network[4].bias]},
    {'params': [H_network[6].bias]},
    # C networks
    {'params': [C_network[0].bias]},
    {'params': [C_network[2].bias]},
    {'params': [C_network[4].bias]},
    {'params': [C_network[6].bias]},
    # N networks
    {'params': [N_network[0].bias]},
    {'params': [N_network[2].bias]},
    {'params': [N_network[4].bias]},
    {'params': [N_network[6].bias]},
    # O networks
    {'params': [O_network[0].bias]},
    {'params': [O_network[2].bias]},
    {'params': [O_network[4].bias]},
    {'params': [O_network[6].bias]}
], lr=1e-3)

AdamW_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(AdamW, factor=0.5, patience=100, threshold=0)
SGD_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(SGD, factor=0.5, patience=100, threshold=0)

# Training
## Validation

In [ ]:
def validate(dataset_):
    # run validation
    mse = torch.nn.MSELoss(reduction='none')
    energy_mse = 0.0
    force_mse = 0.0
    # hessian_mse = 0.0
    hvp_mse = 0.0
    n_molec = 0
    
    for properties in dataset_:
        # Save the properties in variables
        species = properties['species'].to(device)
        coordinates = properties['coordinates'].to(device).requires_grad_(True)
        true_energies = properties['energies'].to(device)
        true_forces = properties['forces'].to(device)
        true_hessian = properties['hessian'].to(device)
        
        # Predict energies from our model
        _, predicted_energies = model((species, coordinates))
        
        # Calculate the number of molecules in the minibatch and the number of atoms in each molecule
        n_molec += predicted_energies.shape[0] # The number of molecules is equal to the number of elements in the energy tensor
        Ns = (species >= 0).sum(dim=1, dtype=torch.int64) # N is a tensor with the number of atoms in each molecule

        # Calculate the HVPs
        vectors = torch.zeros((len(Ns), 3*torch.max(Ns)), device=device) # Initialize tensor that will contain the random vectors `v`

        # Use a vector with i.i.d. values from a Gaussian distribution with zero mean and unit deviation
        for i in range(len(Ns)): # For each system,
            N = Ns[i]               # Get the number of atoms
            values = torch.randn(3*N, device=device) # Create a vector with random values
            vectors[i][:3*N] = values
        vectors = vectors.view(len(Ns), torch.max(Ns), 3)

        """
        # (Optional) Use one-hot vectors for one-column estimator
        for i in range(len(Ns)): # For each system,
            N = Ns[i]               # Get the number of atoms
            column_idx = torch.randint(0, 3*N, (1,)) # Create a random integer from 0 to 3N inclusive
            vectors[i][column_idx] = 1.0
        vectors = vectors.view(len(Ns), torch.max(Ns), 3)
        """

        # Calculate atomic forces
        forces = -torch.autograd.grad(predicted_energies.sum(), coordinates, create_graph=True, retain_graph=True)[0]

        # Calculate HVP using torch.autograd.grad
        hvp = -torch.autograd.grad(forces , coordinates, grad_outputs=vectors, create_graph=True, retain_graph=True)[0]

        # Calculate the predicted Hessian from our model
        # hessian = torchani.utils.hessian(coordinates, forces=forces, retain_graph=True)
        
        vectors_flat = vectors.flatten(start_dim=1).unsqueeze(-1) # For vectors with dimensions (batch_size, 3N, 1)
        true_hvp = torch.bmm(true_hessian,vectors_flat).squeeze(-1)
        true_hvp = true_hvp.view(true_hvp.shape[0], torch.max(Ns), 3)

        # Check if the dot product of the complete Hessian with the random vectors is equal to the predicted HVP
        # assert torch.allclose(torch.bmm(hessian,vectors_flat).squeeze(-1), hvp.view(hvp.shape[0],-1))

        # Sum the Mean Squared Errors from the energies, forces and Hessian tensors
        energy_mse += (mse(predicted_energies, true_energies)).sum().item()
        force_mse += (mse(forces, true_forces).sum(dim=(1,2)) / (3*Ns)).sum().item()
        # hessian_mse += (mse(hessian, true_hessian).sum(dim=(1,2)) / (9*Ns**2)).sum().item() # For full Hessian methods
        # hvp_mse += (mse(hvp, true_hvp).sum(dim=(1,2)) / (3*Ns)).sum().item() # For one-column methods
        hvp_mse += (mse(hvp, true_hvp).sum(dim=(1,2)) / (9*Ns**2)).sum().item() # For Hutchinson methods
        
    # Return the RMSE
    return hartree2kcalmol(math.sqrt(energy_mse / n_molec)), \
           hartree2kcalmol(math.sqrt(force_mse / n_molec)), \
           hartree2kcalmol(math.sqrt(hvp_mse / n_molec))

        #    hartree2kcalmol(math.sqrt(hessian_mse / n_molec)), \

## Training Configuration

Edit these parameters in the training cell before starting runs:
- training_runs: list of run IDs for repeated experiments.
- data_percentage: training-set percentage(s) to evaluate.
- seed_number: controls reproducibility across runs.

The training cell automatically creates trainings/ and writes checkpoints there.

In [ ]:
# Set number of trainings
training_runs = [1]

# Set percentage of training data
data_percentage = [80]

# Set seeds for reproducibility (seed: 7289038)
seed_number = 7289038
random.seed(seed_number)
np.random.seed(seed_number)
torch.manual_seed(seed_number)

for training_number in training_runs:
    for percentage in data_percentage:
        clear_output(wait=True)
        training_name = 'training_{n}_hvp_{pct}pct'.format(n=training_number, pct=percentage)
        training_dir = 'trainings'
        os.makedirs(training_dir, exist_ok=True)
        
        # Set batch size according to amount of training data
        config['batch_size'] = int(percentage*(400/80))
        print('batch size = {}'.format(config['batch_size']))

        # Initialize wandb to keep track of the training
        wandb.init(project="PHL Training", entity="<USER>", config=config, tags=["PHL training", '{pct} percent training data'.format(pct=percentage), \
                                                                                  "Seed: {}".format(seed_number), "HVP"],
                name=training_name)
        # (id="", resume="must")        for resuming a previous run
        
        print('Run ID: {}'.format(wandb.run.id))

        # Load, shuffle, and split dataset
        training, skip, validation, test = torchani.data.load(dspath, additional_properties=('forces','hessian')
        ).subtract_self_energies(energy_shifter, species_order).species_to_indices(species_order).shuffle().split(percentage/100, 0.8-(percentage/100), 0.1, None)
        print('training = {}, skip = {}, validation= {}, test = {}'.format(percentage/100, 0.8-(percentage/100), 0.1, 1-(percentage/100)-(0.8-(percentage/100))-0.1))

        # Collate training, validation, and test datasets
        training = training.collate(config["batch_size"]).cache()
        validation = validation.collate(50).cache()
        test = test.collate(50).cache()
        
        # Save self atomic energies to file
        torch.save(energy_shifter.self_energies, 
                os.path.join(training_dir, 'sae_training-{n}-hvp-{pct}pct.pt'.format(n=training_number, pct=percentage)))
        
        # Save latest checkpoint file name
        latest_checkpoint = os.path.join(training_dir, training_name + '-latest.pt')

        if os.path.isfile(latest_checkpoint):
            checkpoint = torch.load(latest_checkpoint)
            nn.load_state_dict(checkpoint['nn'])
            AdamW.load_state_dict(checkpoint['AdamW'])
            SGD.load_state_dict(checkpoint['SGD'])
            AdamW_scheduler.load_state_dict(checkpoint['AdamW_scheduler'])
            SGD_scheduler.load_state_dict(checkpoint['SGD_scheduler'])

            model = torchani.nn.Sequential(aev_computer, nn).to(device)
            model = torch.nn.DataParallel(model)
            model.to(device)

        else:
            # Initialize parameters
            def init_params(m):
                if isinstance(m, torch.nn.Linear):
                    torch.nn.init.kaiming_normal_(m.weight, a=1.0)
                    torch.nn.init.zeros_(m.bias)
            nn.apply(init_params)

            model = torchani.nn.Sequential(aev_computer, nn).to(device)
            model = torch.nn.DataParallel(model)
            model.to(device)
            
            # Set optimizers
            AdamW = torch.optim.AdamW([
                # H networks
                {'params': [H_network[0].weight]},
                {'params': [H_network[2].weight]},
                {'params': [H_network[4].weight]},
                {'params': [H_network[6].weight]},
                # C networks
                {'params': [C_network[0].weight]},
                {'params': [C_network[2].weight]},
                {'params': [C_network[4].weight]},
                {'params': [C_network[6].weight]},
                # N networks
                {'params': [N_network[0].weight]},
                {'params': [N_network[2].weight]},
                {'params': [N_network[4].weight]},
                {'params': [N_network[6].weight]},
                # O networks
                {'params': [O_network[0].weight]},
                {'params': [O_network[2].weight]},
                {'params': [O_network[4].weight]},
                {'params': [O_network[6].weight]},
            ])

            SGD = torch.optim.SGD([
                # H networks
                {'params': [H_network[0].bias]},
                {'params': [H_network[2].bias]},
                {'params': [H_network[4].bias]},
                {'params': [H_network[6].bias]},
                # C networks
                {'params': [C_network[0].bias]},
                {'params': [C_network[2].bias]},
                {'params': [C_network[4].bias]},
                {'params': [C_network[6].bias]},
                # N networks
                {'params': [N_network[0].bias]},
                {'params': [N_network[2].bias]},
                {'params': [N_network[4].bias]},
                {'params': [N_network[6].bias]},
                # O networks
                {'params': [O_network[0].bias]},
                {'params': [O_network[2].bias]},
                {'params': [O_network[4].bias]},
                {'params': [O_network[6].bias]},
            ], lr=1e-3)
            
            # Set learning rate schedulers
            AdamW_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(AdamW, factor=0.5, patience=100, threshold=0)
            SGD_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(SGD, factor=0.5, patience=100, threshold=0)
        
        
        # Start training
        print("training starting from epoch", AdamW_scheduler.last_epoch + 1)
        mse = torch.nn.MSELoss(reduction='none')
        max_epochs = config['max_epochs']
        early_stopping_learning_rate = 1.0E-5
        force_coefficient = 0.30  # controls the importance of energy loss vs force loss
        hessian_coefficient = 0.09  # controls the importance of energy loss vs Hessian loss
        best_model_checkpoint = os.path.join(training_dir, training_name + '-best.pt')

        times = []
        for _ in range(AdamW_scheduler.last_epoch + 1, max_epochs):


            e_rmse, fc_rmse, hvp_rmse = validate(validation)

            print('Energy RMSE:', e_rmse, 'Force RMSE:', fc_rmse, 'and HVP RMSE:', hvp_rmse, 'at epoch', \
                AdamW_scheduler.last_epoch + 1)

            learning_rate = AdamW.param_groups[0]['lr']

            if learning_rate < early_stopping_learning_rate:
                break

            # checkpoint
            if e_rmse < AdamW_scheduler.best:
                torch.save(nn.state_dict(), best_model_checkpoint)

            AdamW_scheduler.step(e_rmse)
            SGD_scheduler.step(e_rmse)

            wandb.log({'validation_energy_rmse':e_rmse, 'validation_forces_rmse':fc_rmse,
                    'validation_hvp_rmse':hvp_rmse, 'best_validation_energy_rmse': AdamW_scheduler.best, 
                    'learning_rate': learning_rate}, 
                    step = AdamW_scheduler.last_epoch)

            e_rmse, fc_rmse, hvp_rmse = validate(test)
            wandb.log({'test_energy_rmse':e_rmse, 'test_forces_rmse':fc_rmse, 'test_hvp_rmse':hvp_rmse}, 
                    step = AdamW_scheduler.last_epoch)

            torch.cuda.synchronize()
            start_time = time.perf_counter()

            # Epoch starts here
            for i, properties in tqdm(
                enumerate(training),
                total=len(training),
                desc="epoch {}".format(AdamW_scheduler.last_epoch)
            ):


                species = properties['species'].to(device)
                coordinates = properties['coordinates'].to(device).requires_grad_(True)
                true_energies = properties['energies'].to(device)
                true_forces = properties['forces'].to(device)
                true_hessian = properties['hessian'].to(device)
                num_atoms = (species >= 0).sum(dim=1, dtype=torch.int64)

                # Energy predictions
                _, predicted_energies = model((species, coordinates))

                # Calculate the HVPs
                vectors = torch.zeros((len(num_atoms), 3*torch.max(num_atoms)), device=device) # Initialize tensor that will contain the random vectors `v`

                # Use a vector with i.i.d. values from a Gaussian distribution with zero mean and unit deviation
                for i in range(len(num_atoms)): # For each system,
                    N = num_atoms[i]               # Get the number of atoms
                    values = torch.randn(3*N, device=device) # Create a vector with random values
                    vectors[i][:3*N] = values
                vectors = vectors.view(len(num_atoms), torch.max(num_atoms), 3)

                """
                # (Optional) Use one-hot vectors for one-column estimator
                for i in range(len(num_atoms)): # For each system,
                    N = num_atoms[i]               # Get the number of atoms
                    column_idx = torch.randint(0, 3*N, (1,)) # Create a random integer from 0 to 3N inclusive
                    vectors[i][column_idx] = 1.0
                vectors = vectors.view(len(num_atoms), torch.max(num_atoms), 3)
                """

                # Calculate atomic forces
                forces = -torch.autograd.grad(predicted_energies.sum(), coordinates, create_graph=True, retain_graph=True)[0]

                # Calculate HVP using torch.autograd.grad
                hvp = -torch.autograd.grad(forces , coordinates, grad_outputs=vectors, create_graph=True, retain_graph=True)[0]

                # Calculate the Hessian with the original method
                # hessian = torchani.utils.hessian(coordinates, forces=forces, retain_graph=True, create_graph=False)

                # Calculate reference HVP from the reference Hessians
                vectors_flat = vectors.flatten(start_dim=1).unsqueeze(-1) # For vectors with dimensions (batch_size, 3N, 1)
                true_hvp = torch.bmm(true_hessian,vectors_flat).squeeze(-1) # Batched matrix multiplication of Hessians and random vectors
                true_hvp = true_hvp.view(true_hvp.shape[0], torch.max(num_atoms), 3) # Reshape HVP to (batch_size, N, 3)

                # Check if the dot product of the complete Hessian with the random vectors is equal to the predicted HVP
                # assert torch.allclose(torch.bmm(hessian,vectors_flat).squeeze(-1), hvp.view(hvp.shape[0],-1))

                # The total loss has three parts: energy loss, force loss, and Hessian loss
                energy_loss = (mse(predicted_energies, true_energies) / torch.sqrt(num_atoms)).mean()
                force_loss = (mse(forces, true_forces).sum(dim=(1, 2)) / (3*num_atoms)).mean()
                # hessian_loss = (mse(true_hessian, hessian).sum(dim=(1, 2)) / (9*num_atoms**2)).mean() # For full Hessian methods
                hvp_loss = (mse(hvp, true_hvp).sum(dim=(1, 2)) / (9*num_atoms**2)).mean() # For Hutchinson estimator methods
                # hvp_loss = (mse(hvp, true_hvp).sum(dim=(1, 2)) / (3*num_atoms)).mean() # For one-column methods

                #loss = energy_loss + force_coefficient * force_loss + hessian_coefficient * hessian_loss
                loss = energy_loss + force_coefficient * force_loss + hessian_coefficient * hvp_loss

                AdamW.zero_grad()
                SGD.zero_grad()
                loss.backward()
                AdamW.step()
                SGD.step()

            torch.cuda.synchronize()
            end_time = time.perf_counter()
            times.append(end_time-start_time)
            
            torch.save({
                'nn': nn.state_dict(),
                'AdamW': AdamW.state_dict(),
                'SGD': SGD.state_dict(),
                'AdamW_scheduler': AdamW_scheduler.state_dict(),
                'SGD_scheduler': SGD_scheduler.state_dict(),
            }, latest_checkpoint)

        avg_time = np.mean(times)
        print('The average time per epoch is', avg_time, 's')
        wandb.finish()

## Output Artifacts

After training, expected outputs include:
- trainings/sae_training-\<run\>-hvp-\<pct\>pct.pt
- trainings/training_\<run\>\_hvp_\<pct\>pct-latest.pt
- trainings/training_\<run\>\_hvp_\<pct\>pct-best.pt
- W&B run history with validation/test RMSE metrics.